# Module 03 — GPU Memory Hierarchy Experiments

This notebook measures tensor allocation, effective streaming bandwidth, and memory-access layout on an NVIDIA T4. It also calculates simple LLM weight and KV-cache memory estimates.

In [ ]:
import statistics
import time

import torch

# All experiments below require a CUDA device; this notebook targets one T4.
assert torch.cuda.is_available(), "Select a GPU runtime."
device = torch.device("cuda:0")
props = torch.cuda.get_device_properties(device)
print("GPU:", props.name)
print("VRAM:", round(props.total_memory / 2**30, 2), "GiB")

## Experiment A — Tensor memory and allocator behavior

In [ ]:
def show_memory():
    # Allocated counts live tensor storage; reserved also includes cached
    # blocks kept by PyTorch's allocator for future allocations.
    allocated = torch.cuda.memory_allocated(device) / 2**20
    reserved = torch.cuda.memory_reserved(device) / 2**20
    print(f"Allocated: {allocated:.2f} MiB")
    print(f"Reserved:  {reserved:.2f} MiB")

show_memory()
# 4096 x 4096 FP32 values require 4096 * 4096 * 4 bytes = 64 MiB.
x = torch.empty((4096, 4096), dtype=torch.float32, device=device)
print("Logical tensor memory:", x.numel() * x.element_size() / 2**20, "MiB")
show_memory()

del x
# Deleting the tensor releases the live allocation, but the caching
# allocator may keep the block reserved until it is explicitly trimmed.
print("After deleting tensor:")
show_memory()
torch.cuda.empty_cache()
print("After empty_cache():")
show_memory()

## Experiment B — Effective global-memory bandwidth

For `C = A + B` with FP32 tensors, the useful traffic estimate is 12 bytes per element: one read for A, one read for B, and one write for C.

In [ ]:
elements = 8_000_000
repeats = 50
A = torch.randn(elements, device=device)
B = torch.randn(elements, device=device)
C = torch.empty_like(A)

def add_kernel():
    # Logical traffic is two reads (A and B) plus one write (C).
    torch.add(A, B, out=C)

# Warm up before timing to remove first-use setup from the median.
for _ in range(10):
    add_kernel()
torch.cuda.synchronize(device)

times_ms = []
for _ in range(repeats):
    # CUDA events measure elapsed device time rather than Python launch time.
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    add_kernel()
    end.record()
    end.synchronize()
    times_ms.append(start.elapsed_time(end))

median_ms = statistics.median(times_ms)
# This is useful logical traffic, not a direct DRAM-counter measurement.
useful_bytes = 3 * elements * A.element_size()
effective_gbs = useful_bytes / (median_ms / 1000) / 1e9
print(f"Useful traffic estimate: {useful_bytes / 1e6:.2f} MB")
print(f"Median time: {median_ms:.4f} ms")
print(f"Effective useful-data rate: {effective_gbs:.2f} GB/s")

## Experiment C — Contiguous versus strided access

This optional experiment uses Numba CUDA. It compares neighboring threads reading neighboring values with neighboring threads reading values separated by a stride.

In [ ]:
try:
    import numpy as np
    from numba import cuda

    @cuda.jit
    def gather_kernel(src, dst, stride, size):
        # Larger strides separate neighboring threads' addresses and may
        # require less efficient memory transactions.
        index = cuda.grid(1)
        # Protect the final partially filled block.
        if index < size:
            dst[index] = src[index * stride]

    n = 1_000_000
    source = cuda.to_device(np.arange(n * 32, dtype=np.float32))
    destination = cuda.device_array(n, dtype=np.float32)
    threads = 256
    blocks = (n + threads - 1) // threads

    for stride in (1, 2, 4, 8, 16, 32):
        # Warm up each stride so Numba compilation does not affect timing.
        gather_kernel[blocks, threads](source, destination, stride, n)
        cuda.synchronize()
        samples = []
        for _ in range(30):
            # Synchronize because a kernel launch is asynchronous.
            start = time.perf_counter()
            gather_kernel[blocks, threads](source, destination, stride, n)
            cuda.synchronize()
            samples.append((time.perf_counter() - start) * 1000)
        print(f"Stride {stride:2d}: {statistics.median(samples):.4f} ms")
except ImportError:
    print("Install numba, numba-cuda, and numpy to run this experiment.")

## LLM memory estimates

In [ ]:
parameter_count = 1_000_000_000
# FP16 uses two bytes per parameter; this is weight storage only.
fp16_weight_bytes = parameter_count * 2
print(f"1B FP16 weights: {fp16_weight_bytes / 2**30:.2f} GiB")

layers = 32
kv_heads = 8
head_dim = 128
context_tokens = 2048
bytes_per_value = 2
# The leading factor of two accounts for both K and V tensors.
kv_bytes = 2 * layers * kv_heads * head_dim * context_tokens * bytes_per_value
print(f"One sequence KV cache: {kv_bytes / 2**20:.2f} MiB")
print(f"Twenty sequences KV cache: {20 * kv_bytes / 2**30:.2f} GiB")

## Interpretation

Compare the measured bandwidth with the T4’s published peak, then explain the gap. Relate the result to batch-one decode, weight reads, KV-cache growth, and vLLM’s need to manage GPU memory carefully.